# Gemma Serial Depth Analysis

This notebook analyzes the serial depth of different Gemma models across various sequence lengths.

In [ ]:
#@title Imports

import jax
import numpy as np
import matplotlib.pyplot as plt

!pip install gemma
from gemma import gm
from serial_depth import depth

print(gm.nn.Gemma2_2B)

import time
from typing import Dict, List, Tuple

In [ ]:
#@title Configuration

# All Gemma models to test
MODELS = [
    'Gemma2_2B',
    'Gemma2_9B',
    'Gemma2_27B',
    'Gemma3_1B',
    'Gemma3_4B',
    'Gemma3_12B',
    'Gemma3_27B'
]

# Sequence lengths to test (powers of 2)
BASE_SEQUENCE_LENGTHS = [2**i for i in [3, 5, 7, 9, 11, 13, 15]]  # 8 to 32768
EXTENDED_SEQUENCE_LENGTHS = BASE_SEQUENCE_LENGTHS + [2**17]  # Add 131072 for 27B models

# Models that can handle the extended sequence length
EXTENDED_MODELS = ['Gemma2_27B', 'Gemma3_4B', 'Gemma3_12B', 'Gemma3_27B']

print(f"Testing {len(MODELS)} models with sequence lengths: {BASE_SEQUENCE_LENGTHS}")
print(f"Extended models ({EXTENDED_MODELS}) will also test: {2**17}")

In [ ]:
#@title Experiment helper functions

def load_model_and_compute_depth(model_name: str, sequence_length: int, verbose: bool = False) -> int:
    """Load a Gemma model and compute its serial depth for given sequence length.

    Args:
        model_name: Name of the Gemma model (e.g., 'Gemma3_1B')
        sequence_length: Sequence length to use for computation
        verbose: Whether to enable verbose output

    Returns:
        The computed serial depth
    """

    # Load the model class
    model_class = getattr(gm.nn, model_name)
    model = model_class()
    rng = jax.random.PRNGKey(0)

    # Create dummy input tensor
    vocab_size = model.config.num_embed
    tokens = np.random.randint(0, vocab_size, size=(1, sequence_length))
    abstract_tokens = jax.ShapeDtypeStruct(tokens.shape, tokens.dtype)

    # Get abstract variables and jaxpr
    abstract_vars = jax.eval_shape(lambda: model.init(rng, tokens=abstract_tokens))
    closed_jaxpr = jax.make_jaxpr(model.apply)(abstract_vars, tokens=abstract_tokens)
    jaxpr = closed_jaxpr.jaxpr

    # Compute depth
    result = depth.compute_depth(jaxpr, verbose=verbose)

    return result

def run_experiments() -> Dict[str, List[Tuple[int, int]]]:
    """Run serial depth computation experiments for all models and sequence lengths.

    Returns:
        Dictionary mapping model names to list of (sequence_length, depth) tuples
    """
    results = {}

    for model_name in MODELS:
        print(f"\nTesting {model_name}...")
        model_results = []

        # Determine sequence lengths for this model
        seq_lengths = EXTENDED_SEQUENCE_LENGTHS if model_name in EXTENDED_MODELS else BASE_SEQUENCE_LENGTHS

        for seq_len in seq_lengths:
            try:
                print(f"  Sequence length {seq_len}...", end=" ")
                start_time = time.time()

                computed_depth = load_model_and_compute_depth(model_name, seq_len)

                elapsed = time.time() - start_time
                print(f"depth={computed_depth} ({elapsed:.1f}s)")

                model_results.append((seq_len, computed_depth))

            except Exception as e:
                print(f"ERROR: {e}")
                continue

        results[model_name] = model_results
        print(f"Completed {len(model_results)} computation for {model_name}")

    return results

## Run Experiments

In [ ]:
# Run the experiments
print("Starting depth analysis experiments...")
experiment_results = run_experiments()
print("\nExperiments completed!")

## Visualization

In [ ]:
# Create the plot
plt.figure(figsize=(12, 8))

# Color palette for different models
colors = plt.cm.tab10(np.linspace(0, 1, len(MODELS)))

for i, (model_name, results) in enumerate(experiment_results.items()):
    if not results:  # Skip if no results
        continue

    seq_lengths, depths = zip(*results)

    plt.plot(seq_lengths, depths, 'o-',
             color=colors[i], label=model_name,
             linewidth=2, markersize=6)

plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0), useMathText=True)

plt.xscale('log', base=2)
plt.xlabel('Sequence Length (tokens)', fontsize=12)
plt.ylabel(f'Serial Depth', fontsize=12)
plt.title('Serial Depth vs Sequence Length for Gemma Models', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')



# Set x-axis ticks to powers of 2
all_seq_lengths = set()
for results in experiment_results.values():
    for seq_len, _ in results:
        all_seq_lengths.add(seq_len)

plt.xticks(sorted(all_seq_lengths))
#plt.xticks(rotation=45)

plt.tight_layout()
plt.show()